In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
mrwellsdavid_unsw_nb15_path = kagglehub.dataset_download('mrwellsdavid/unsw-nb15')

print('Data source import complete.')


# Data

https://www.kaggle.com/datasets/mrwellsdavid/unsw-nb15/data

# Index

1. Data Inspection
2. Data Preprocessing for EDA
3. Exploratory Data Analysis (EDA)
4. Data Preprocessing for Modelling
5. Modelling

# 1. Data Inspection

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

In [ ]:
NB15_features = pd.read_csv('/kaggle/input/unsw-nb15/NUSW-NB15_features.csv', encoding='cp1252')
NB15_features

In [ ]:
df = pd.read_csv('/kaggle/input/unsw-nb15/UNSW_NB15_training-set.csv')

In [ ]:
df

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.describe(include='object')

# 2. Data Preprocessing for EDA

In [ ]:
df.isna().sum()

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.isna().sum().sum()

In [ ]:
df.drop(['id'], axis=1, inplace=True)

In [ ]:
# Replacing '-' in state and service for 'other'
df['state'] = df['state'].replace('-','other')
df['service'] = df['service'].replace('-','other')

# 3. EDA

In [ ]:
counts = df['label'].value_counts()
fig = px.pie(counts, values=counts.values, names=counts.index, title='Distribution of Label')
fig.show()

Grouping by attack category

In [ ]:
df[df['attack_cat']!='normal'].groupby('attack_cat').size()

We need to group Backdoor and Backdoors into a single category

In [ ]:
df['attack_cat'] = df['attack_cat'].apply(lambda item: 'Backdoor' if item =='Backdoors' else item)

In [ ]:
attack_by_cat = df[df['attack_cat']!='normal'].groupby('attack_cat').size().reset_index(name='counts')
attack_by_cat

Distribution of attacks by category

In [ ]:
attack_by_cat = df[df['attack_cat']!='normal'].groupby('attack_cat').size().reset_index(name='counts').sort_values(by='counts', ascending = False)
fig = px.bar(attack_by_cat, x='attack_cat', y='counts', title='Distribution of attack categories')
fig.show()

Distribution of source bytes by attack category

In [ ]:
plt.figure(figsize=(10,8))
ax = sns.kdeplot(x='sbytes',hue='attack_cat',log_scale=True,data=df[df['attack_cat']!='normal'])
ax.set_title('Distribution source bytes by attack category')
ax.set_xlabel('Source to destination transaction bytes')
ax.set_ylabel('Density')
plt.xticks(rotation=45)
plt.show()

Most used services by attack category

In [ ]:
df_attacks_cat_services = df[(df['attack_cat']!='normal')].groupby(['attack_cat','service']).size().reset_index(name='Count')

In [ ]:
# Create a 3x3 subplot grid
fig = make_subplots(rows=3, cols=3,
                    subplot_titles=('Category: Analysis', 'Category: Backdoor', 'Category: DoS',
                                    'Category: Exploits', 'Category: Fuzzers', 'Category: Generic',
                                    'Category: Reconnaissance', 'Category: Shellcode', 'Category: Worms'))

# Create a list of attack categories to loop through
categories = ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Reconnaissance', 'Shellcode', 'Worms']

# Define row and column positions for each plot
positions = [(1,1), (1,2), (1,3), (2,1), (2,2), (2,3), (3,1), (3,2), (3,3)]

# Loop through the categories and create a bar plot for each one
for i, category in enumerate(categories):
    # Filter the dataframe for each attack category
    df_filtered = df_attacks_cat_services[df_attacks_cat_services['attack_cat'] == category]

    # Add bar plot to the subplot
    fig.add_trace(go.Bar(x=df_filtered['service'], y=df_filtered['Count'], name=category,
                         marker_color=px.colors.qualitative.Plotly[i]),
                  row=positions[i][0], col=positions[i][1])

# Update layout
fig.update_layout(height=900, width=900, title_text="Service Counts by Attack Categories", showlegend=False)

# Update x-axis for all subplots
fig.update_xaxes(tickangle=90)

# Show the figure
fig.show()


Top-5 protocols used by attack category

In [ ]:
df_attacks_cat_proto = df[(df['attack_cat']!='normal')].groupby(['attack_cat','proto']).size().reset_index(name='Count')

In [ ]:
# Create a 3x3 subplot grid
fig = make_subplots(rows=3, cols=3,
                    subplot_titles=('Category: Analysis', 'Category: Backdoor', 'Category: DoS',
                                    'Category: Exploits', 'Category: Fuzzers', 'Category: Generic',
                                    'Category: Reconnaissance', 'Category: Shellcode', 'Category: Worms'))

# Create a list of attack categories to loop through
categories = ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Reconnaissance', 'Shellcode', 'Worms']

# Define row and column positions for each plot
positions = [(1,1), (1,2), (1,3), (2,1), (2,2), (2,3), (3,1), (3,2), (3,3)]

# Loop through the categories and create a bar plot for each one (top 5 protocols)
for i, category in enumerate(categories):
    # Filter the dataframe for each attack category and take the top 5 protocols
    df_filtered = df_attacks_cat_proto[df_attacks_cat_proto['attack_cat'] == category][:5]

    # Add bar plot to the subplot
    fig.add_trace(go.Bar(x=df_filtered['proto'], y=df_filtered['Count'], name=category,
                         marker_color=px.colors.qualitative.Plotly[i]),
                  row=positions[i][0], col=positions[i][1])

# Update layout
fig.update_layout(height=900, width=900, title_text="Top 5 Protocols by Attack Categories", showlegend=False)

# Update x-axis for all subplots
fig.update_xaxes(tickangle=90)

# Show the figure
fig.show()


Distribution of states by attack category

In [ ]:
df_attacks_cat_state = df[(df['attack_cat']!='normal')].groupby(['attack_cat','state']).size().reset_index(name='Count')

In [ ]:
# Create a 3x3 subplot grid
fig = make_subplots(rows=3, cols=3,
                    subplot_titles=('Category: Analysis', 'Category: Backdoor', 'Category: DoS',
                                    'Category: Exploits', 'Category: Fuzzers', 'Category: Generic',
                                    'Category: Reconnaissance', 'Category: Shellcode', 'Category: Worms'))

# List of attack categories
categories = ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Reconnaissance', 'Shellcode', 'Worms']

# Define the row and column positions for the plots
positions = [(1,1), (1,2), (1,3), (2,1), (2,2), (2,3), (3,1), (3,2), (3,3)]

# Loop through the attack categories and create a bar plot for each one
for i, category in enumerate(categories):
    # Filter the dataframe for each attack category
    df_filtered = df_attacks_cat_state[df_attacks_cat_state['attack_cat'] == category]

    # Add bar plot to the subplot
    fig.add_trace(go.Bar(x=df_filtered['state'], y=df_filtered['Count'], name=category,
                         marker_color=px.colors.qualitative.Plotly[i]),
                  row=positions[i][0], col=positions[i][1])

# Update the layout of the entire figure
fig.update_layout(height=900, width=900, title_text="Attack Counts by State and Category", showlegend=False)

# Rotate x-axis labels for better readability
fig.update_xaxes(tickangle=90)

# Show the plot
fig.show()


**Summary of EDA:**


*   The distribution of attacks types is not uniform, most of them are generic.
*   The distributions of source bytes sent by attack type follow a similar trend, there are not noticeable differences between them.
*   Most of the attacks use HTTP service (not considering 'other' category).
*   3pc and argus are protocols commonly used in distributed computing environments. Both are used by the majority of attacking types along with TCP and UDP
*   INT and FIN are the the states in which most connections are.






In [ ]:
numerical_columns = [col for col in df.columns if df[col].dtypes != "object"]
# Calculate the correlation matrix for numerical columns
corr = df[numerical_columns].corr()

plt.figure(figsize=(30, 25))
sns.heatmap(corr, annot=True, cmap='coolwarm', linewidths=0.5, fmt='.2f')
plt.title('Correlation Heatmap between Numeric Features')
plt.show()

In [ ]:
corr_matrix = df[numerical_columns].corr().abs()
upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_pairs = upper_triangle.stack().reset_index()
high_corr_pairs.columns = ['Feature 1', 'Feature 2', 'Correlation']
high_corr_pairs = high_corr_pairs[high_corr_pairs['Correlation'] > 0.80]
high_corr_pairs

In [ ]:
features_to_drop = ['ct_state_ttl', 'ct_dst_sport_ltm', 'ct_src_ltm']
df = df.drop(features_to_drop, axis=1)

# 4. Preprocessing for the Model

#### Encoding

In [ ]:
categorical_columns = df.select_dtypes(include=['object']).columns

In [ ]:
categorical_columns

In [ ]:
df = pd.get_dummies(df, columns=['proto', 'service'], drop_first=True)

In [ ]:
label_encoder = LabelEncoder()
df['state'] = label_encoder.fit_transform(df['state'])
df['attack_cat'] = label_encoder.fit_transform(df['attack_cat'])

#### Feature Engineering

In [ ]:
# Interaction between source and destination load
df['load_interaction'] = df['sload'] * df['dload']

In [ ]:
# Total transaction bytes between source and destination
df['total_bytes'] = df['sbytes'] + df['dbytes']

In [ ]:
# Packet flow ratio between source and destination
df['pkt_flow_ratio'] = df['spkts'] / (df['dpkts'] + 1)

In [ ]:
# Bytes difference and ratio
df['bytes_diff'] = df['sbytes'] - df['dbytes']
df['bytes_ratio'] = df['sbytes'] / (df['dbytes'] + 1)

In [ ]:
# TTL difference
df['ttl_diff'] = df['sttl'] - df['dttl']

In [ ]:
# Jitter difference and ratio
df['jitter_diff'] = df['sjit'] - df['djit']
df['jitter_ratio'] = df['sjit'] / (df['djit'] + 1)

In [ ]:
# Difference between synack and ackdat times
df['tcp_time_diff'] = df['synack'] - df['ackdat']

In [ ]:
df.sample(10)

In [ ]:
data1 =df.copy()
data2 =df.copy()

# 5. Modelling

## Data 1 -> Without Over Sampling

In [ ]:
from sklearn.model_selection import train_test_split
X = data1.drop('label', axis=1)
y = data1['label']

In [ ]:
scalers = [
    MinMaxScaler(),
    RobustScaler()
]

In [ ]:
!pip install xgboost
import xgboost as xgb # Import the xgboost library

In [ ]:
models = [
    ('KNN', KNeighborsClassifier(), {}),
    ('Random Forest', RandomForestClassifier(), {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10]
    }),
    ('XGBoost', xgb.XGBClassifier(eval_metric='mlogloss'), {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.01, 0.1, 0.2],
        'max_depth': [3, 6, 9]
    })
]

In [ ]:
def summarize_metrics(model_name,y_train,t_test,y_train_predict,y_test_predict, existing_df=None):
    """
    This method is used to generate classification report for both train and test and generate a dataframe that summarizes different
    models metrics, it takes the precited values for each model and scaling method name, model name, and an existing dataframe to append
    different results
    """

    scaler_names = {
        'MinMaxScaler': 'MinMaxScaler',
        'StandardScaler': 'StandardScaler',
        'RobustScaler': 'RobustScaler',
    }

    # Determine the name of the scaler
    scaler_name = scaler_names.get(type(scaler).__name__, 'No Scaling')


    # Generating classification reports
    train_report_dict = classification_report(y_train, y_train_pred, output_dict=True)
    test_report_dict = classification_report(y_test, y_test_pred, output_dict=True)

    # Extracting and rounding metrics for train
    accuracy_train = round(train_report_dict['accuracy'], 2)
    macro_avg_train = train_report_dict['macro avg']
    precision_train = round(macro_avg_train['precision'], 2)
    recall_train = round(macro_avg_train['recall'], 2)
    f1_train = round(macro_avg_train['f1-score'], 2)

    # Extracting and rounding metrics for test
    accuracy_test = round(test_report_dict['accuracy'], 2)
    macro_avg_test = test_report_dict['macro avg']
    precision_test = round(macro_avg_test['precision'], 2)
    recall_test = round(macro_avg_test['recall'], 2)
    f1_test = round(macro_avg_test['f1-score'], 2)

    # Create a summary dictionary
    summary_dict = {
        'Model': model_name,
        'Scaling Method': scaler_name,
        'Train Accuracy': accuracy_train,
        'Test Accuracy': accuracy_test,
        'Train Precision': precision_train,
        'Test Precision': precision_test,
        'Train Recall': recall_train,
        'Test Recall': recall_test,
        'Train F1-Score': f1_train,
        'Test F1-Score': f1_test
    }

    summary_df = pd.DataFrame([summary_dict])

    # Append to existing DataFrame or return new DataFrame
    if existing_df is not None:
        return pd.concat([existing_df, summary_df], ignore_index=True)
    else:
        return summary_df


In [ ]:
def tune_model(model, param_grid, X_train, y_train, X_test, y_test):
    """
    This function takes a model and a parameter grid, performs hyperparameter tuning using gridsearch.
    and returns the best esimator
    """
    # Fitting the data with GridSearchCV
    grid_search = GridSearchCV(model, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_ # Best model

    # Predictions
    y_train_pred = best_model.predict(X_train)
    y_test_pred = best_model.predict(X_test)
    print("Values for class", y_train.value_counts())

    # ============== Evaluation ===============
    train_accuracy = accuracy_score(y_train, y_train_pred)
    test_accuracy = accuracy_score(y_test, y_test_pred)
    print("Best Parameters:", grid_search.best_params_)
    print(f"Training Accuracy: {train_accuracy:.4f}")
    print(f"Test Accuracy: {test_accuracy:.4f}")

    return best_model

In [ ]:
results_df = None

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

def plot_confusion_matrix(y_true, y_pred, title):
    conf_matrix = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(8, 6))
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(title)
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()

imputer = SimpleImputer(strategy='mean')

for scaler in scalers:
    # ========= scaling ==========
    X_scaled = scaler.fit_transform(X)
    X_scaled = imputer.fit_transform(X_scaled)

    # Train Test Split
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42, stratify=y)

    for model_name, model, param_grid in models:
        print(f"Running model: {model_name} with {scaler.__class__.__name__}")

        # ============== Hyperparameter Tuning (Finding best parameters) ==========
        best_model = tune_model(model, param_grid, X_train, y_train, X_test, y_test)

        # Predict using best model
        y_train_pred = best_model.predict(X_train)
        y_test_pred = best_model.predict(X_test)

        # ======== Summarize and Evaluate ==============
        results_df = summarize_metrics(
            model_name,
            y_train, y_test,
            y_train_pred, y_test_pred,
            existing_df=results_df
        )

        # ======== Visualize Confusion Matrices ==============
        plot_confusion_matrix(y_test, y_test_pred, title=f"Test Confusion Matrix for {model_name}")
        print("\n=======================================================\n")

                # ======== Print Classification Report ==============
        print(f"Classification Report for {model_name} on Test Set")
        print(classification_report(y_test, y_test_pred))

        print("\n=======================================================\n")

In [ ]:
results_df

## Data 2 -> Smote

In [ ]:
from sklearn.model_selection import train_test_split
X = data2.drop('label', axis=1)
y = data2['label']

In [ ]:
scalers = [
    MinMaxScaler(),
    StandardScaler()
]

In [ ]:
!pip install xgboost
import xgboost as xgb # Import the xgboost library

In [ ]:
models = [
    ('KNN', KNeighborsClassifier(), {}),
    ('Random Forest', RandomForestClassifier(), {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10]
    }),
    ('XGBoost', xgb.XGBClassifier(eval_metric='mlogloss'), {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.01, 0.1, 0.2],
        'max_depth': [3, 6, 9]
    })
]

In [ ]:
def summarize_metrics(model_name,y_train,t_test,y_train_predict,y_test_predict, existing_df=None):
    """
    This method is used to generate classification report for both train and test and generate a dataframe that summarizes different
    models metrics, it takes the precited values for each model and scaling method name, model name, and an existing dataframe to append
    different results
    """

    scaler_names = {
        'MinMaxScaler': 'MinMaxScaler',
        'StandardScaler': 'StandardScaler',
        'RobustScaler': 'RobustScaler',
    }

    # Determine the name of the scaler
    scaler_name = scaler_names.get(type(scaler).__name__, 'No Scaling')


    # Generating classification reports
    train_report_dict = classification_report(y_train, y_train_pred, output_dict=True)
    test_report_dict = classification_report(y_test, y_test_pred, output_dict=True)

    # Extracting and rounding metrics for train
    accuracy_train = round(train_report_dict['accuracy'], 2)
    macro_avg_train = train_report_dict['macro avg']
    precision_train = round(macro_avg_train['precision'], 2)
    recall_train = round(macro_avg_train['recall'], 2)
    f1_train = round(macro_avg_train['f1-score'], 2)

    # Extracting and rounding metrics for test
    accuracy_test = round(test_report_dict['accuracy'], 2)
    macro_avg_test = test_report_dict['macro avg']
    precision_test = round(macro_avg_test['precision'], 2)
    recall_test = round(macro_avg_test['recall'], 2)
    f1_test = round(macro_avg_test['f1-score'], 2)

    # Create a summary dictionary
    summary_dict = {
        'Model': model_name,
        'Scaling Method': scaler_name,
        'Train Accuracy': accuracy_train,
        'Test Accuracy': accuracy_test,
        'Train Precision': precision_train,
        'Test Precision': precision_test,
        'Train Recall': recall_train,
        'Test Recall': recall_test,
        'Train F1-Score': f1_train,
        'Test F1-Score': f1_test
    }

    summary_df = pd.DataFrame([summary_dict])

    # Append to existing DataFrame or return new DataFrame
    if existing_df is not None:
        return pd.concat([existing_df, summary_df], ignore_index=True)
    else:
        return summary_df


In [ ]:
def tune_model(model, param_grid, X_train, y_train, X_test, y_test):
    """
    This function takes a model and a parameter grid, performs hyperparameter tuning using gridsearch.
    and returns the best esimator
    """
    # Fitting the data with GridSearchCV
    grid_search = GridSearchCV(model, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_ # Best model

    # Predictions
    y_train_pred = best_model.predict(X_train)
    y_test_pred = best_model.predict(X_test)
    print("Values for class", y_train.value_counts())

    # ============== Evaluation ===============
    train_accuracy = accuracy_score(y_train, y_train_pred)
    test_accuracy = accuracy_score(y_test, y_test_pred)
    print("Best Parameters:", grid_search.best_params_)
    print(f"Training Accuracy: {train_accuracy:.4f}")
    print(f"Test Accuracy: {test_accuracy:.4f}")

    return best_model

In [ ]:
results_df = None

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from imblearn.over_sampling import SMOTE

def plot_confusion_matrix(y_true, y_pred, title):
    conf_matrix = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(8, 6))
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(title)
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()

imputer = SimpleImputer(strategy='mean')

for scaler in scalers:
    # ========= scaling ==========
    X_scaled = scaler.fit_transform(X)
    X_scaled = imputer.fit_transform(X_scaled)

    # Train Test Split
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42, stratify=y)

    # Initialize SMOTE
    smote = SMOTE(random_state=42)

    # Fit and resample your training data
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

    for model_name, model, param_grid in models:
        print(f"Running model: {model_name} with {scaler.__class__.__name__}")

        # ============== Hyper Paramater Tunning (Finding best parameters) ==========
        best_model = tune_model(model, param_grid, X_train_resampled, y_train_resampled, X_test, y_test)
        # Predict using best model
        y_train_pred = best_model.predict(X_train_resampled)
        y_test_pred = best_model.predict(X_test)

        # ======== Summarize and Evaluate ==============
        results_df = summarize_metrics(
            model_name,
            y_train_resampled, y_test,
            y_train_pred, y_test_pred,
            existing_df=results_df
        )

        # ======== Visualize Confusion Matrices ==============
        plot_confusion_matrix(y_test, y_test_pred, title=f"Test Confusion Matrix for {model_name}")
        print("\n=======================================================\n")

In [ ]:
results_df

# Thank You